# 02 — LLM Insight Generation

Validate the IONOS LLM integration for generating growth insights:
- Load analytics data from previous notebook
- Call IONOS Llama 3.3 with insight prompt
- Iterate on prompt quality
- Validate structured JSON output

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

IONOS_API_TOKEN = os.getenv('IONOS_API_TOKEN')
print(f'IONOS token loaded: {"yes" if IONOS_API_TOKEN else "NO — add to .env"}')

## 1. Load Analytics Data

Load the insights from `01_umami_ingest` notebook output.

In [ ]:
from agent.storage import LocalStorage

store = LocalStorage(base_dir='../state')
insights_raw = store.read('insights.json')

if insights_raw is None:
    print('ERROR: Run 01_umami_ingest.ipynb first to generate insights.json')
else:
    print(f'Loaded insights: {insights_raw["website_analytics"]["visitors"]} visitors')
    print(f'Top pages: {len(insights_raw["website_analytics"]["top_pages"])}')
    print(f'Funnels: {list(insights_raw["website_analytics"]["event_funnels"].keys())}')

## 2. Connect to IONOS LLM

In [ ]:
from agent.llm_client import LLMClient

llm = LLMClient(api_token=IONOS_API_TOKEN)

# Quick test
test = llm.chat([
    {'role': 'user', 'content': 'Say hello in exactly 5 words.'}
])
print(f'LLM response: {test["content"]}')
print(f'Tokens used: {test["usage"]}')

## 3. Insight Generation Prompt

Send analytics data to the LLM and ask for growth opportunities.

In [ ]:
import json

# Load strategy defaults
from agent.models import Strategy
strategy = Strategy()

insight_prompt = f"""You are a social media growth analyst for a technical blog (fretchen.eu).

The blog covers: {', '.join(strategy.content_pillars)}
Social channels: {', '.join(strategy.channels)}
Target audience: {strategy.target_audience}

Here is the website analytics data from the last 7 days:

Summary:
- Pageviews: {insights_raw['website_analytics']['pageviews']}
- Unique visitors: {insights_raw['website_analytics']['visitors']}
- Visits: {insights_raw['website_analytics']['visits']}
- Bounces: {insights_raw['website_analytics']['bounces']}

Top pages:
{json.dumps(insights_raw['website_analytics']['top_pages'][:10], indent=2)}

Top referrers:
{json.dumps(insights_raw['website_analytics']['top_referrers'][:10], indent=2)}

Tracked events (user engagement funnels):
{json.dumps(insights_raw['website_analytics']['top_events'][:10], indent=2)}

Based on this data, identify:
1. Which blog topics have the most visitor interest?
2. Where is traffic coming from? Any social media referrals?
3. Which pages would make the best social media content to share?
4. What content gaps exist — popular topics with no recent social posts?
5. Suggest 3-5 specific, actionable growth opportunities for Mastodon and Bluesky.

Return your analysis as JSON with this structure:
{{
  "top_topics": ["topic1", "topic2"],
  "traffic_sources": ["source1", "source2"],
  "best_pages_for_social": [{{"path": "/...", "reason": "..."}}],
  "content_gaps": ["gap1", "gap2"],
  "growth_opportunities": ["opportunity1", "opportunity2"]
}}

Return ONLY the JSON, no additional text."""

print(f'Prompt length: {len(insight_prompt)} chars')
print('---')
print(insight_prompt[:500] + '...')

In [ ]:
# Call LLM with insight prompt
result = llm.chat(
    messages=[
        {'role': 'system', 'content': 'You are a data-driven social media growth analyst. Return only valid JSON.'},
        {'role': 'user', 'content': insight_prompt}
    ],
    temperature=0.3,  # Low temperature for structured output
    max_tokens=2048,
)

print(f'Tokens used: {result["usage"]}')
print(f'Model: {result["model"]}')
print('---')
print(result['content'])

## 4. Parse and Validate JSON Output

In [ ]:
# Try to parse the LLM output as JSON
content = result['content'].strip()

# Strip markdown code fences if present
if content.startswith('```'):
    content = content.split('\n', 1)[1]  # Remove first line
    content = content.rsplit('```', 1)[0]  # Remove last fence

try:
    analysis = json.loads(content)
    print('\n=== Parsed Analysis ===')
    print(f'\nTop topics: {analysis.get("top_topics", [])}')
    print(f'\nTraffic sources: {analysis.get("traffic_sources", [])}')
    print(f'\nBest pages for social:')
    for page in analysis.get('best_pages_for_social', []):
        print(f'  - {page.get("path", "?")} — {page.get("reason", "?")}')
    print(f'\nContent gaps: {analysis.get("content_gaps", [])}')
    print(f'\nGrowth opportunities:')
    for opp in analysis.get('growth_opportunities', []):
        print(f'  - {opp}')
except json.JSONDecodeError as e:
    print(f'ERROR: Could not parse JSON: {e}')
    print(f'Raw output: {content[:500]}')

## 5. Update Insights with Growth Opportunities

In [ ]:
from datetime import datetime
from agent.models import Insights

# Update insights with LLM analysis
insights = Insights(**insights_raw)
insights.growth_opportunities = analysis.get('growth_opportunities', [])
insights.last_analysis = datetime.now()

# Save enriched insights
store.write('insights.json', insights)

# Also save raw LLM analysis for reference
store.write('llm_analysis.json', analysis)

print(f'Saved {len(insights.growth_opportunities)} growth opportunities')
print(f'Analysis timestamp: {insights.last_analysis}')

In [ ]:
llm.close()
print('Done — LLM insight generation validated.')